In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset, IterableDataset
import sys
print(torch.cuda.is_available())
sys.path.append("/mnt/home/lserrano/disco-ball/")

import numpy as np
import random
import matplotlib.pyplot as plt
import h5py
import os
from tqdm import tqdm
from einops import rearrange
from torch.func import functional_call, vmap


In [ ]:
from models import DISCOHouse, vectors_to_parameters
from advection_diffusion import Fractaloid
from train import DISCOLitModule, advection_diffusion_analytical
from utils import RelativeL2
from plot_dataset_samples import plot_prediction_vs_ground_truth

In [ ]:
def autoregressive_predict(model, initial_seq, n_pred, device):
    preds = []
    current = initial_seq.clone().to(device)
    n_input = current.shape[1]
    for t in range(n_pred):
        inp = current[:, -n_input:].to(device)
        with torch.no_grad():
            state_labels = torch.tensor([0], device=inp.device)
            next_frame, metadata = model(inp, state_labels, n_future_steps=1)
            if t == 0:
                theta = metadata['theta_latent']
        current = torch.cat([current, next_frame], axis=1)
        preds.append(next_frame)
    return torch.cat(preds, axis=1), theta

In [ ]:
class TemporalBatchDatasetFly(IterableDataset):
    def __init__(self, n_batches, batch_size, sub_x, sub_t, split="train", input_frames=16, output_frames=2,
                 L=16.0, nx=256, nt=100, T=10.0,
                 v_range=(0.01, 1.0), D_range=(0.01, 1.0),
                 fractal_degree=8, fractal_power=2, seed=None):
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.L = L
        self.nx = nx
        self.nt = nt
        self.T = T
        self.v_range = v_range
        self.D_range = D_range
        self.fractal_degree = fractal_degree
        self.fractal_power = fractal_power
        self.seed = seed
        self.rng = np.random.default_rng(seed)

    def __iter__(self):
        for _ in range(self.n_batches):
            input_frames = self.input_frames
            batch_inputs = []
            batch_targets = []
            batch_v = []
            batch_d = []
            batch_init = []
            for _ in range(self.batch_size):
                # Sample advection speed and viscosity
                if self.split == 'train':
                    if random.random() < 0.5:
                        v = self.rng.uniform(*self.v_range) if isinstance(self.v_range, (tuple, list)) else float(self.v_range)
                        D = 0
                    else:
                        v = 0
                        D = self.rng.uniform(*self.D_range) if isinstance(self.D_range, (tuple, list)) else float(self.D_range)
                else:
                    v = self.rng.uniform(*self.v_range) if isinstance(self.v_range, (tuple, list)) else float(self.v_range)
                    D = self.rng.uniform(*self.D_range) if isinstance(self.D_range, (tuple, list)) else float(self.D_range)
                # Generate fractaloid initial condition
                fractaloid = Fractaloid(
                    degree=self.fractal_degree,
                    power=self.fractal_power,
                    size=self.nx,
                    patch_size=self.nx
                )
                u0 = fractaloid.generate(batch_size=1, seed=None).squeeze(0).numpy()
                u0 = (u0 - u0.mean()) / (u0.std() + 1e-8)
                u_xt, x, t = advection_diffusion_analytical(
                    u0, L=self.L, v=v, D=D, nt=self.nt, T=self.T
                )
                u_xt = u_xt[::self.sub_t, ::self.sub_x]
                max_start_index_input = u_xt.shape[0] - input_frames
                input = u_xt[:input_frames].copy()
                target = u_xt[input_frames: input_frames + self.output_frames].copy()
                batch_inputs.append(torch.from_numpy(input).unsqueeze(-2).float())
                batch_targets.append(torch.from_numpy(target).unsqueeze(-2).float())
                batch_v.append(v)
                batch_d.append(D)
                batch_init.append(torch.from_numpy(u0))
            batch = {
                'input': torch.stack(batch_inputs),
                'target': torch.stack(batch_targets),
                'velocities': batch_v,
                'diffusivities': batch_d,
                'initial_conditions': torch.stack(batch_init)
            }
            yield batch

In [ ]:
batch_size=128
sub_x=1
sub_t=1
n_input_frames=16
n_output_frames=1 #50-n_input_frames

In [ ]:
n_batches = int(128*10//batch_size)  # or set as needed for your epoch size
split="train"
train_ds = TemporalBatchDatasetFly(
    n_batches=n_batches,
    batch_size=batch_size,
    sub_x=sub_x,
    sub_t=sub_t,
    split=split,
    input_frames=n_input_frames,
    output_frames=n_output_frames,
    L=16.0,
    nx=256,
    nt=100,
    T=10.0,
    fractal_power=2.0,
    fractal_degree=256, # nx
    v_range=(0.01, 1.0),
    D_range=(0.001, 1.0),
)
train_loader = DataLoader(train_ds, batch_size=None, num_workers=1, prefetch_factor=1, pin_memory=True)

In [ ]:
#theta_path = "/mnt/home/lserrano/disco-ball/results/advection_diffusion/dense"
device="cuda" if torch.cuda.is_available() else "cpu"
#ckpt_time="2025-06-24/09-30-29" # full space of advection x diffusion
#ckpt_time="2025-07-01/00-21-40"#in-context #2025-06-24/15-27-28" no in-context
ckpt_time="2025-07-09/22-13-33" # in-context and no bias
ckpt_path = f"/mnt/home/lserrano/disco-ball/outputs/{ckpt_time}/model_final.ckpt"
#theta_path = "/mnt/home/lserrano/disco-ball/results/advection_diffusion/dense"
print(f"Loading model from {ckpt_path}...")
model = DISCOLitModule.load_from_checkpoint(ckpt_path, map_location=device)
model = model.model.to(device)
model.eval()

# I. Train

In [ ]:
results_dir = f"results/{ckpt_time}"
dataset_name="advection_diffusion"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/plots/", exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/predictions/", exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/theta/", exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/errors/", exist_ok=True)
#output_path = f"plots/{ckpt_time}"
#output_path = f"plots/{setting}/baseline/"

In [ ]:
relative_l2_error = RelativeL2()

In [ ]:
num_samples_to_save = 3
random_indices = sorted(random.sample(range(n_batches), min(num_samples_to_save, n_batches)))
batch_idx = 0


n=0
rollout_error=0
all_theta=[]
all_velocities = []
all_diffusivities = []

for batch in tqdm(train_loader):
    inp, target = batch["input"], batch["target"]
    all_velocities += batch["velocities"]
    all_diffusivities += batch["diffusivities"]
    
    inp = inp.to(device)
    target = target.to(device)
    n_sample = inp.shape[0]
    pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)
    rollout_error += relative_l2_error(pred, target).item()*n_sample
    n += n_sample
    all_theta.append(theta)
    # Save predictions and ground truth for selected batches
    if batch_idx in random_indices:
        for i in range(min(3, inp.shape[0])):  # Save up to 3 samples per batch
            pred_np = pred[i].detach().cpu().numpy()
            target_np = target[i].detach().cpu().numpy()
            out_dir = f"{results_dir}/{dataset_name}/predictions/"
            os.makedirs(out_dir, exist_ok=True)
            np.savez_compressed(f"{out_dir}/pred_gt_batch{batch_idx}_sample{i}.npz", pred=pred_np, gt=target_np)
            plot_prediction_vs_ground_truth(pred_np, target_np, idx=i, out_dir=out_dir, split=split, sample_name=f"batch{batch_idx}")
    batch_idx += 1

all_labels = [split] * len(all_theta)
all_theta = torch.cat(all_theta, dim=0)
print("Error next time-step:", rollout_error/n)

# 1. General plot

In [ ]:
theta = all_theta.detach().cpu().numpy()
velocities = np.array(all_velocities)
diffusivities = np.array(all_diffusivities)

In [ ]:
pivot="velocity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta[:, 0], theta[:, 1], c=color, alpha=0.5, s=8)
#plt.scatter(theta[:, 0], theta[:, 1], alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", "theta_vs_velocity.png"))
plt.show()
plt.close()

In [ ]:
pivot="diffusivity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta[:, 0], theta[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"theta_vs_{pivot}.png"))
plt.show()
plt.close()

# 2. Zoom on pure advection

In [ ]:
mask = diffusivities==0

In [ ]:
pivot="velocity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta[mask, 0], theta[mask, 1], c=color[mask], alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"focus_theta_vs_{pivot}.png"))
plt.show()
plt.close()

# 3. Zoom on pure diffusion

In [ ]:
mask = velocities ==0

In [ ]:
pivot="diffusivity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta[mask, 0], theta[mask, 1], c=color[mask], alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"focus_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
# 4. import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler # Good practice for PCA


In [ ]:
scaler = StandardScaler()
theta_scaled = scaler.fit_transform(theta)

# Initialize PCA - for 2D data, we'll keep 2 components
pca = PCA(n_components=2)

# Fit PCA to the data and transform it
theta_pca = pca.fit_transform(theta_scaled)

In [ ]:
pivot="velocity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta_pca[:, 0], theta_pca[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
pivot="diffusivity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta_pca[:, 0], theta_pca[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
# analyze theta and biases 

In [ ]:
all_theta.device

In [ ]:
dim=1
all_theta_expanded = model.decode_theta(all_theta, dim)

In [ ]:
param_dict = dict(model.opnns[str(dim)].named_parameters())
batched_params_dict = vectors_to_parameters(all_theta_expanded, param_dict)

In [ ]:
batched_params_dict["adapter_in.linear.weight"].shape

In [ ]:
n_output_frames

# II. Test 

In [ ]:
split="test"
n_batches = int(1000//batch_size)  # or set as needed for your epoch size
test_ds = TemporalBatchDatasetFly(
    n_batches=n_batches,
    batch_size=batch_size,
    sub_x=sub_x,
    sub_t=sub_t,
    split=split,
    input_frames=n_input_frames,
    output_frames=n_output_frames,
    L=16.0,
    nx=256,
    nt=100,
    T=10.0,
    fractal_power=2.0,
    fractal_degree=256, # nx
    v_range=(0.01, 1.0),
    D_range=(0.001, 1.0),
)
test_loader = DataLoader(test_ds, batch_size=None, num_workers=4, prefetch_factor=4, pin_memory=True)

In [ ]:
num_samples_to_save = 3
random_indices = sorted(random.sample(range(n_batches), min(num_samples_to_save, n_batches)))
batch_idx = 0

n=0
rollout_error=0
all_theta=[]
all_velocities = []
all_diffusivities = []

for batch in tqdm(test_loader):
    inp, target = batch["input"], batch["target"]
    all_velocities += batch["velocities"]
    all_diffusivities += batch["diffusivities"]
    
    inp = inp.to(device)
    target = target.to(device)
    n_sample = inp.shape[0]
    pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)
    rollout_error += relative_l2_error(pred, target).item()*n_sample
    n += n_sample
    all_theta.append(theta)
    # Save predictions and ground truth for selected batches
    if batch_idx in random_indices:
        for i in range(min(3, inp.shape[0])):  # Save up to 3 samples per batch
            pred_np = pred[i].detach().cpu().numpy()
            target_np = target[i].detach().cpu().numpy()
            out_dir = f"{results_dir}/{dataset_name}/predictions/"
            os.makedirs(out_dir, exist_ok=True)
            np.savez_compressed(f"{out_dir}/pred_gt_batch{batch_idx}_sample{i}.npz", pred=pred_np, gt=target_np)
            plot_prediction_vs_ground_truth(pred_np, target_np, idx=i, out_dir=out_dir, split=split, sample_name=f"batch{batch_idx}")
    batch_idx += 1

all_labels = [split] * len(all_theta)
all_theta = torch.cat(all_theta, dim=0)
print("Test error next time-step:", rollout_error/n)

In [ ]:
theta = all_theta.detach().cpu().numpy()
velocities = np.array(all_velocities)
diffusivities = np.array(all_diffusivities)

In [ ]:
pivot="velocity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta[:, 0], theta[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
pivot="diffusivity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta[:, 0], theta[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
pivot="velocity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta[mask, 0], theta[mask, 1], c=color[mask], alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
import matplotlib.cm as cm # Import colormap module

# --- Define Diffusivity Tranches and Colors ---
# Define the bins for diffusivity tranches
# Note: The last bin is inclusive (<=) for the upper bound
diffusivity_tranches = [
    (0.001, 0.01, "0.001 - 0.01"),
    (0.01, 0.05, "0.01 - 0.05"),
    (0.05, 0.1, "0.05 - 0.1"),
    (0.1, 0.2, "0.1 - 0.2"),
    (0.2, 0.5, "0.2 - 0.5"),
    (0.5, 1.0, "0.5 - 1.0") # Adjust upper bound to be <= 1.0
]

# Choose a colormap for distinct colors
# 'viridis' is a good perceptual colormap
colors = cm.get_cmap('viridis', len(diffusivity_tranches))

# --- Plotting ---
plt.figure(figsize=(10, 8))

up = 1.0
down = 0.75

# For this example, I'll apply the velocity filter and then tranche the diffusivity broadly.
overall_velocity_mask = (velocities < up) & (velocities >= down)

print(f"Samples after overall velocity mask: {np.sum(overall_velocity_mask)}")

# Iterate through tranches and plot
for i, (lower_bound, upper_bound, label) in enumerate(diffusivity_tranches):
    # Create a mask for the current diffusivity tranche AND the overall velocity mask
    tranche_mask = (diffusivities >= lower_bound) & \
                   (diffusivities <= upper_bound) & \
                   overall_velocity_mask # Apply the velocity mask

    if np.sum(tranche_mask) > 0: # Only plot if there are points in this tranche
        plt.scatter(
            theta[tranche_mask, 0],
            theta[tranche_mask, 1],
            c=[colors(i)], # Use the color from the colormap for this tranche
            alpha=0.6,
            s=10, # Slightly larger points for clarity
            label=f"Diffusivity: {label}" # Label for the legend
        )
        print(f"Tranche '{label}' has {np.sum(tranche_mask)} points.")
    else:
        print(f"Tranche '{label}' has no points matching the criteria.")


plt.xlabel("theta_latent[0]", fontsize=12)
plt.ylabel("theta_latent[1]", fontsize=12)
plt.title(f"Theta Latent 2D Projection by Diffusivity Tranches for {down} <= velocity <={up}", fontsize=14)
plt.legend(title="Diffusivity Ranges", bbox_to_anchor=(1.05, 1), loc='upper left') # Place legend outside
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to prevent legend overlap
plt.show()
plt.close()

In [ ]:
import matplotlib.cm as cm # Import colormap module

# --- Define Diffusivity Tranches and Colors ---
# Define the bins for diffusivity tranches
# Note: The last bin is inclusive (<=) for the upper bound
diffusivity_tranches = [
    (0.001, 0.01, "0.001 - 0.01"),
    (0.01, 0.05, "0.01 - 0.05"),
    (0.05, 0.1, "0.05 - 0.1"),
    (0.1, 0.2, "0.1 - 0.2"),
    (0.2, 0.5, "0.2 - 0.5"),
    (0.5, 1.0, "0.5 - 1.0") # Adjust upper bound to be <= 1.0
]

# Choose a colormap for distinct colors
# 'viridis' is a good perceptual colormap
colors = cm.get_cmap('viridis', len(diffusivity_tranches))

# --- Plotting ---
plt.figure(figsize=(10, 8))

up = 0.75
down = 0.5

# For this example, I'll apply the velocity filter and then tranche the diffusivity broadly.
overall_velocity_mask = (velocities < up) & (velocities >= down)

print(f"Samples after overall velocity mask: {np.sum(overall_velocity_mask)}")

# Iterate through tranches and plot
for i, (lower_bound, upper_bound, label) in enumerate(diffusivity_tranches):
    # Create a mask for the current diffusivity tranche AND the overall velocity mask
    tranche_mask = (diffusivities >= lower_bound) & \
                   (diffusivities <= upper_bound) & \
                   overall_velocity_mask # Apply the velocity mask

    if np.sum(tranche_mask) > 0: # Only plot if there are points in this tranche
        plt.scatter(
            theta[tranche_mask, 0],
            theta[tranche_mask, 1],
            c=[colors(i)], # Use the color from the colormap for this tranche
            alpha=0.6,
            s=10, # Slightly larger points for clarity
            label=f"Diffusivity: {label}" # Label for the legend
        )
        print(f"Tranche '{label}' has {np.sum(tranche_mask)} points.")
    else:
        print(f"Tranche '{label}' has no points matching the criteria.")


plt.xlabel("theta_latent[0]", fontsize=12)
plt.ylabel("theta_latent[1]", fontsize=12)
plt.title(f"Theta Latent 2D Projection by Diffusivity Tranches for {down} <= velocity <={up}", fontsize=14)
plt.legend(title="Diffusivity Ranges", bbox_to_anchor=(1.05, 1), loc='upper left') # Place legend outside
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to prevent legend overlap
plt.show()
plt.close()

In [ ]:
import matplotlib.cm as cm # Import colormap module

# --- Define Diffusivity Tranches and Colors ---
# Define the bins for diffusivity tranches
# Note: The last bin is inclusive (<=) for the upper bound
diffusivity_tranches = [
    (0.001, 0.01, "0.001 - 0.01"),
    (0.01, 0.05, "0.01 - 0.05"),
    (0.05, 0.1, "0.05 - 0.1"),
    (0.1, 0.2, "0.1 - 0.2"),
    (0.2, 0.5, "0.2 - 0.5"),
    (0.5, 1.0, "0.5 - 1.0") # Adjust upper bound to be <= 1.0
]

# Choose a colormap for distinct colors
# 'viridis' is a good perceptual colormap
colors = cm.get_cmap('viridis', len(diffusivity_tranches))

# --- Plotting ---
plt.figure(figsize=(10, 8))

up = 0.5
down = 0.25

# For this example, I'll apply the velocity filter and then tranche the diffusivity broadly.
overall_velocity_mask = (velocities < up) & (velocities >= down)

print(f"Samples after overall velocity mask: {np.sum(overall_velocity_mask)}")

# Iterate through tranches and plot
for i, (lower_bound, upper_bound, label) in enumerate(diffusivity_tranches):
    # Create a mask for the current diffusivity tranche AND the overall velocity mask
    tranche_mask = (diffusivities >= lower_bound) & \
                   (diffusivities <= upper_bound) & \
                   overall_velocity_mask # Apply the velocity mask

    if np.sum(tranche_mask) > 0: # Only plot if there are points in this tranche
        plt.scatter(
            theta[tranche_mask, 0],
            theta[tranche_mask, 1],
            c=[colors(i)], # Use the color from the colormap for this tranche
            alpha=0.6,
            s=10, # Slightly larger points for clarity
            label=f"Diffusivity: {label}" # Label for the legend
        )
        print(f"Tranche '{label}' has {np.sum(tranche_mask)} points.")
    else:
        print(f"Tranche '{label}' has no points matching the criteria.")


plt.xlabel("theta_latent[0]", fontsize=12)
plt.ylabel("theta_latent[1]", fontsize=12)
plt.title(f"Theta Latent 2D Projection by Diffusivity Tranches for {down} <= velocity <={up}", fontsize=14)
plt.legend(title="Diffusivity Ranges", bbox_to_anchor=(1.05, 1), loc='upper left') # Place legend outside
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to prevent legend overlap
plt.show()
plt.close()

In [ ]:
import matplotlib.cm as cm # Import colormap module

# --- Define Diffusivity Tranches and Colors ---
# Define the bins for diffusivity tranches
# Note: The last bin is inclusive (<=) for the upper bound
diffusivity_tranches = [
    (0.001, 0.01, "0.001 - 0.01"),
    (0.01, 0.05, "0.01 - 0.05"),
    (0.05, 0.1, "0.05 - 0.1"),
    (0.1, 0.2, "0.1 - 0.2"),
    (0.2, 0.5, "0.2 - 0.5"),
    (0.5, 1.0, "0.5 - 1.0") # Adjust upper bound to be <= 1.0
]

# Choose a colormap for distinct colors
# 'viridis' is a good perceptual colormap
colors = cm.get_cmap('viridis', len(diffusivity_tranches))

# --- Plotting ---
plt.figure(figsize=(10, 8))

up = 0.25
down = 0.

# For this example, I'll apply the velocity filter and then tranche the diffusivity broadly.
overall_velocity_mask = (velocities < up) & (velocities >= down)

print(f"Samples after overall velocity mask: {np.sum(overall_velocity_mask)}")

# Iterate through tranches and plot
for i, (lower_bound, upper_bound, label) in enumerate(diffusivity_tranches):
    # Create a mask for the current diffusivity tranche AND the overall velocity mask
    tranche_mask = (diffusivities >= lower_bound) & \
                   (diffusivities <= upper_bound) & \
                   overall_velocity_mask # Apply the velocity mask

    if np.sum(tranche_mask) > 0: # Only plot if there are points in this tranche
        plt.scatter(
            theta[tranche_mask, 0],
            theta[tranche_mask, 1],
            c=[colors(i)], # Use the color from the colormap for this tranche
            alpha=0.6,
            s=10, # Slightly larger points for clarity
            label=f"Diffusivity: {label}" # Label for the legend
        )
        print(f"Tranche '{label}' has {np.sum(tranche_mask)} points.")
    else:
        print(f"Tranche '{label}' has no points matching the criteria.")


plt.xlabel("theta_latent[0]", fontsize=12)
plt.ylabel("theta_latent[1]", fontsize=12)
plt.title(f"Theta Latent 2D Projection by Diffusivity Tranches for {down} <= velocity <={up}", fontsize=14)
plt.legend(title="Diffusivity Ranges", bbox_to_anchor=(1.05, 1), loc='upper left') # Place legend outside
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to prevent legend overlap
plt.show()
plt.close()

In [ ]:
scaler = StandardScaler()
theta_scaled = scaler.fit_transform(theta)

# Initialize PCA - for 2D data, we'll keep 2 components
pca = PCA(n_components=2)

# Fit PCA to the data and transform it
theta_pca = pca.fit_transform(theta_scaled)

In [ ]:
pivot="velocity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta_pca[:, 0], theta_pca[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
pivot="diffusivity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta_pca[:, 0], theta_pca[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

# III. Further analysis

## III.a Parameters over time

In [ ]:
split="train"
n_batches = int(128//batch_size)  # or set as needed for your epoch size
n_input_frames=50
test_ds = TemporalBatchDatasetFly(
    n_batches=n_batches,
    batch_size=batch_size,
    sub_x=sub_x,
    sub_t=sub_t,
    split=split,
    input_frames=50,
    output_frames=n_output_frames,
    L=16.0,
    nx=256,
    nt=100,
    T=10.0,
    fractal_power=2.0,
    fractal_degree=256, # nx
    v_range=0.1,
    D_range=0,
)
test_loader = DataLoader(test_ds, batch_size=None, num_workers=4, prefetch_factor=4, pin_memory=True)

In [ ]:
num_samples_to_save = 3
random_indices = sorted(random.sample(range(n_batches), min(num_samples_to_save, n_batches)))
batch_idx = 0

n=0
rollout_error=0
all_theta=[]
all_velocities = []
all_diffusivities = []

for batch in tqdm(test_loader):
    inp, target = batch["input"], batch["target"]
    all_velocities += batch["velocities"]
    all_diffusivities += batch["diffusivities"]
    
    inp = inp.to(device)
    target = target.to(device)
    n_sample = inp.shape[0]
    theta = []
    for t in range(50-16):
        pred, theta_t = autoregressive_predict(model, inp[:, t:t+16], n_pred=target.shape[1], device=device)
        theta.append(theta_t)
    theta = torch.stack(theta)
    print('theta.shape', theta.shape)
    all_theta.append(theta)

all_labels = [split] * len(all_theta)
all_theta = torch.cat(all_theta, 1)


In [ ]:
all_theta = all_theta.detach().cpu().numpy()

In [ ]:
i=15
theta_index = all_theta[:,i]
pivot="viscosity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.scatter(theta_index[:, 0], theta_index[:, 1], c=[t for t in range(theta_index.shape[0])], alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
i=1
theta_index = all_theta[:,i]
pivot="diffusivity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.plot(theta_index[:, 1])
plt.legend()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
i=3
theta_index = all_theta[:,i]
pivot="diffusivity"
color = diffusivities if pivot=="diffusivity" else velocities
plt.plot(theta_index[:, 1])
plt.legend()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
plt.show()
plt.close()

## III.b Fixed initial conditions, varying parameters

In [ ]:
class TemporalDatasetFixedCI(torch.utils.data.Dataset):
    def __init__(self, n_batches, batch_size, sub_x, sub_t, split="train", input_frames=16, output_frames=2,
                 L=16.0, nx=256, nt=100, T=10.0,
                 v_range=[0.01, 0.025, 0.05, 0.1, 0.5, 1.0], D_range=[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0],
                 fractal_degree=8, fractal_power=2, seed=None):
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.L = L
        self.nx = nx
        self.nt = nt
        self.T = T
        self.v_range = v_range
        self.D_range = D_range
        self.fractal_degree = fractal_degree
        self.fractal_power = fractal_power
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.u0 = []
        for _ in range(self.batch_size):
            fractaloid = Fractaloid(
                degree=self.fractal_degree,
                power=self.fractal_power,
                size=self.nx,
                patch_size=self.nx
            )
            u0 = fractaloid.generate(batch_size=1, seed=None).squeeze(0).numpy()
            u0 = (u0 - u0.mean()) / (u0.std() + 1e-8)
            self.u0.append(torch.from_numpy(u0))
            
        self.u0 = torch.stack(self.u0)

    def __len__(self):
        return len(self.u0)

    def __getitem__(self, idx):
    
        input_frames = self.input_frames
        batch_inputs = []
        batch_targets = []
        batch_v = []
        batch_d = []
        batch_init = []
        for v in self.v_range:
            for d in self.D_range:
                u0 = self.u0[idx]
                u_xt, x, t = advection_diffusion_analytical(
                    u0, L=self.L, v=v, D=d, nt=self.nt, T=self.T
                )
                u_xt = u_xt[::self.sub_t, ::self.sub_x]
                input = u_xt[:input_frames].copy()
                target = u_xt[input_frames: input_frames + self.output_frames].copy()
                
                batch_inputs.append(torch.from_numpy(input).unsqueeze(-2).float())
                batch_targets.append(torch.from_numpy(target).unsqueeze(-2).float())
                batch_v.append(v)
                batch_d.append(d)
                batch_init.append(u0)
                
        batch = {
            'input': torch.stack(batch_inputs),
            'target': torch.stack(batch_targets),
            'velocities': batch_v,
            'diffusivities': batch_d,
            'initial_conditions': torch.stack(batch_init)
        }
        return batch

In [ ]:
split="test"
batch_size=8
n_batches = int(1//batch_size)  # or set as needed for your epoch size

v_range_denser_2 = np.round(np.logspace(np.log10(0.01), np.log10(1), num=20), 3).tolist()
D_range_denser_2 = np.round(np.logspace(np.log10(0.001), np.log10(1.), num=25), 4).tolist()

test_ds = TemporalDatasetFixedCI(
    n_batches=n_batches,
    batch_size=batch_size,
    sub_x=sub_x,
    sub_t=sub_t,
    split=split,
    input_frames=n_input_frames,
    output_frames=n_output_frames,
    L=16.0,
    nx=256,
    nt=100,
    T=10.0,
    v_range=v_range_denser_2,
    D_range=D_range_denser_2,
    fractal_power=2.0,
    fractal_degree=256, # nx
)
test_loader = DataLoader(test_ds, batch_size=1, num_workers=1, prefetch_factor=1, pin_memory=True)

In [ ]:
batch_idx = 0

n=0
rollout_error=0
all_theta=[]
all_velocities = []
all_diffusivities = []

for batch in tqdm(test_loader):
    inp, target = batch["input"], batch["target"]
    inp = inp.squeeze(0) # squeeze the batch dimensions as there are different parameters
    
    all_velocities += batch["velocities"]
    all_diffusivities += batch["diffusivities"]
    
    inp = inp.to(device)
    target = target.to(device)
    n_sample = inp.shape[0]
    pred, theta = autoregressive_predict(model, inp[:, t:t+16], n_pred=target.shape[1], device=device)
    print(theta.shape)
    all_theta.append(theta)

all_labels = [split] * len(all_theta)
all_theta = torch.stack(all_theta)


In [ ]:
results_dir

In [ ]:
all_theta.shape
np.save(f"{results_dir}/advection_diffusion/theta/initial_conditions_vs_parameters.npz", all_theta.detach().cpu())

In [ ]:
i=0
theta_index = all_theta[i].detach().cpu()
pivot="viscosity"
color = torch.tensor(batch['diffusivities']) if pivot=="diffusivity" else torch.tensor(batch['velocities'])
plt.scatter(theta_index[:, 0], theta_index[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
i=0
theta_index = all_theta[i].detach().cpu()
pivot="diffusivity"
color = torch.tensor(batch['diffusivities']) if pivot=="diffusivity" else torch.tensor(batch['velocities'])
plt.scatter(theta_index[:, 0], theta_index[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
i=0
theta_index = all_theta[i].detach().cpu()
pivot="viscosity"
color = torch.tensor(batch['diffusivities']) if pivot=="diffusivity" else torch.tensor(batch['velocities'])
plt.scatter(theta_index[:, 0], theta_index[:, 1], c=color, alpha=0.5, s=8)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label(pivot)
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
import numpy as np

In [ ]:
# Determine the color data based on the 'pivot' variable
i=0
theta_index = all_theta[i].detach().cpu()
z_axis_choice = 0
if z_axis_choice not in [0, 1] or z_axis_choice >= theta_index.shape[1]:
    print(f"Warning: Invalid z_axis_choice ({z_axis_choice}). Defaulting to theta_latent[0].")
    z_axis_choice = 0

z_data = theta_index[:, z_axis_choice].numpy()
z_label = f"theta_latent[{z_axis_choice}]"

# Create the 3D figure and axes
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Create the 3D scatter plot
# Using 'viridis' colormap for clear visualization of the color scale
scatter = ax.scatter(
    batch['velocities'],      # X-axis data
    batch['diffusivities'],     # Y-axis data
    z_data,              # Z-axis data
    c=theta_index[:, abs(z_axis_choice-1)],        # Color points based on the pivot variable
    cmap='viridis',      # Colormap for the 'c' values
    alpha=0.6,           # Transparency of points
    s=20                 # Size of points
)

# Set labels for each axis
ax.set_xlabel("Advection speed")
ax.set_ylabel("Viscosity")
ax.set_zlabel(z_label)

# Add a title to the plot
plt.title(f"3D Projection: Advection vs. Speed vs. {z_label} (Colored by theta_latent[{abs(z_axis_choice-1)}]")

# Add a color bar to show the mapping of colors to the pivot values
cbar = fig.colorbar(scatter, ax=ax, pad=0.1) # 'pad' moves the colorbar slightly away from the plot
cbar.set_label(f"{pivot.capitalize()}") # Capitalize the pivot label for clarity

# Adjust layout to prevent labels/titles from overlapping
plt.tight_layout()

# Display the plot
plt.show()

# Close the plot to free up memory
plt.close()

In [ ]:
# Determine the color data based on the 'pivot' variable
i=0
theta_index = all_theta[i].detach().cpu()
z_axis_choice = 1
if z_axis_choice not in [0, 1] or z_axis_choice >= theta_index.shape[1]:
    print(f"Warning: Invalid z_axis_choice ({z_axis_choice}). Defaulting to theta_latent[0].")
    z_axis_choice = 0

z_data = theta_index[:, z_axis_choice].numpy()
z_label = f"theta_latent[{z_axis_choice}]"

# Create the 3D figure and axes
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Create the 3D scatter plot
# Using 'viridis' colormap for clear visualization of the color scale
scatter = ax.scatter(
    batch['velocities'],      # X-axis data
    batch['diffusivities'],     # Y-axis data
    z_data,              # Z-axis data
    c=theta_index[:, abs(z_axis_choice-1)],        # Color points based on the pivot variable
    cmap='viridis',      # Colormap for the 'c' values
    alpha=0.6,           # Transparency of points
    s=20                 # Size of points
)

# Set labels for each axis
ax.set_xlabel("Advection speed")
ax.set_ylabel("Viscosity")
ax.set_zlabel(z_label)

# Add a title to the plot
plt.title(f"3D Projection: Advection vs. Speed vs. {z_label} (Colored by theta_latent[{abs(z_axis_choice-1)}]")

# Add a color bar to show the mapping of colors to the pivot values
cbar = fig.colorbar(scatter, ax=ax, pad=0.1) # 'pad' moves the colorbar slightly away from the plot
cbar.set_label(f"{pivot.capitalize()}") # Capitalize the pivot label for clarity

# Adjust layout to prevent labels/titles from overlapping
plt.tight_layout()

# Display the plot
plt.show()

# Close the plot to free up memory
plt.close()

In [ ]:
# Determine the color data based on the 'pivot' variable
i=0
theta_index = all_theta[i].detach().cpu()
z_axis_choice = 0
if z_axis_choice not in [0, 1] or z_axis_choice >= theta_index.shape[1]:
    print(f"Warning: Invalid z_axis_choice ({z_axis_choice}). Defaulting to theta_latent[0].")
    z_axis_choice = 0

z_data = theta_index[:, z_axis_choice].numpy()
z_label = f"theta_latent[{z_axis_choice}]"

# Create the 3D figure and axes
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Create the 3D scatter plot
# Using 'viridis' colormap for clear visualization of the color scale
scatter = ax.scatter(
    theta_index[:, 0],      # X-axis data
    theta_index[:, 1],     # Y-axis data
    batch['velocities'],              # Z-axis data
    c=batch['diffusivities'],        # Color points based on the pivot variable
    cmap='viridis',      # Colormap for the 'c' values
    alpha=0.6,           # Transparency of points
    s=20                 # Size of points
)

# Set labels for each axis
ax.set_xlabel("theta_latent[0]")
ax.set_ylabel("theta_latent[1]")
ax.set_zlabel("Advection speed")

# Add a title to the plot
plt.title(f"3D Projection: Advection vs. Speed vs. {z_label} (Colored by viscosity")

# Add a color bar to show the mapping of colors to the pivot values
cbar = fig.colorbar(scatter, ax=ax, pad=0.1) # 'pad' moves the colorbar slightly away from the plot
cbar.set_label(f"{pivot.capitalize()}") # Capitalize the pivot label for clarity

# Adjust layout to prevent labels/titles from overlapping
plt.tight_layout()

ax.view_init(elev=20, azim=200)

# Display the plot
plt.show()

# Close the plot to free up memory
plt.close()

In [ ]:
# Determine the color data based on the 'pivot' variable
i=0
theta_index = all_theta[i].detach().cpu()
z_axis_choice = 0
if z_axis_choice not in [0, 1] or z_axis_choice >= theta_index.shape[1]:
    print(f"Warning: Invalid z_axis_choice ({z_axis_choice}). Defaulting to theta_latent[0].")
    z_axis_choice = 0

z_data = theta_index[:, z_axis_choice].numpy()
z_label = f"theta_latent[{z_axis_choice}]"

# Create the 3D figure and axes
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Create the 3D scatter plot
# Using 'viridis' colormap for clear visualization of the color scale
scatter = ax.scatter(
    theta_index[:, 0],      # X-axis data
    theta_index[:, 1],     # Y-axis data
    batch['diffusivities'],              # Z-axis data
    c=batch['velocities'],        # Color points based on the pivot variable
    cmap='viridis',      # Colormap for the 'c' values
    alpha=0.6,           # Transparency of points
    s=20                 # Size of points
)

# Set labels for each axis
ax.set_xlabel("theta_latent[0]")
ax.set_ylabel("theta_latent[1]")
ax.set_zlabel("Viscosity")

# Add a title to the plot
plt.title(f"3D Projection: Advection vs. Speed vs. {z_label} (Colored by advection speed")

# Add a color bar to show the mapping of colors to the pivot values
cbar = fig.colorbar(scatter, ax=ax, pad=0.1) # 'pad' moves the colorbar slightly away from the plot
cbar.set_label("Advection speed") # Capitalize the pivot label for clarity

# Adjust layout to prevent labels/titles from overlapping
plt.tight_layout()

ax.view_init(elev=10, azim=65)

# Display the plot
plt.show()

# Close the plot to free up memory
plt.close()

## III.c Fixed parameters, varying initial conditions

In [ ]:
all_theta = all_theta.cpu().numpy()

In [ ]:
param_index = 3
#theta_index = all_theta[i].detach().cpu()
n_traj = 2
theta_flatten = all_theta[:n_traj].reshape(-1, 2)
colors = np.arange(n_traj)[:, None].repeat(all_theta.shape[1]).reshape(-1)
pivot="viscosity"
plt.scatter(theta_flatten[:, 0], theta_flatten[:, 1], c=colors, alpha=0.4, s=5)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label("Initial condition")
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
plt.plot(test_ds.u0[0])

In [ ]:
plt.plot(test_ds.u0[1])

In [ ]:
param_index = 3
#theta_index = all_theta[i].detach().cpu()
n_traj = 2
theta_flatten = all_theta[2:4].reshape(-1, 2)
colors = np.arange(n_traj)[:, None].repeat(all_theta.shape[1]).reshape(-1)
pivot="viscosity"
plt.scatter(theta_flatten[:, 0], theta_flatten[:, 1], c=colors, alpha=0.4, s=5)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label("Initial condition")
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
plt.plot(test_ds.u0[2])

In [ ]:
plt.plot(test_ds.u0[3])

In [ ]:

n_traj = 2
theta_flatten = all_theta[4:6].reshape(-1, 2)
colors = np.arange(n_traj)[:, None].repeat(all_theta.shape[1]).reshape(-1)
pivot="viscosity"
plt.scatter(theta_flatten[:, 0], theta_flatten[:, 1], c=colors, alpha=0.4, s=5)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label("Initial condition")
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
plt.plot(test_ds.u0[4])

In [ ]:
plt.plot(test_ds.u0[5])

In [ ]:

n_traj = 2
theta_flatten = all_theta[6:8].reshape(-1, 2)
colors = np.arange(n_traj)[:, None].repeat(all_theta.shape[1]).reshape(-1)
pivot="viscosity"
plt.scatter(theta_flatten[:, 0], theta_flatten[:, 1], c=colors, alpha=0.4, s=5)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label("Initial condition")
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
plt.plot(test_ds.u0[6])

In [ ]:
plt.plot(test_ds.u0[7])

In [ ]:
split="test"
batch_size=128
n_batches = int(1//batch_size)  # or set as needed for your epoch size

test_ds = TemporalDatasetFixedCI(
    n_batches=n_batches,
    batch_size=batch_size,
    sub_x=sub_x,
    sub_t=sub_t,
    split=split,
    input_frames=n_input_frames,
    output_frames=n_output_frames,
    L=16.0,
    nx=256,
    nt=100,
    T=10.0,
    v_range=[0.6], #0.6
    D_range=[0.], #0.2
    fractal_power=2.0,
    fractal_degree=256, # nx
)
test_loader = DataLoader(test_ds, batch_size=128, num_workers=1, prefetch_factor=1, pin_memory=True)

In [ ]:
batch_idx = 0

n=0
rollout_error=0
all_theta=[]
all_velocities = []
all_diffusivities = []

for batch in tqdm(test_loader):
    inp, target = batch["input"], batch["target"]
    inp = inp.squeeze(1) # squeeze the batch dimensions as there are different parameters
    ci = inp[:, 0]
    
    
    all_velocities += batch["velocities"]
    all_diffusivities += batch["diffusivities"]
    
    inp = inp.to(device)
    target = target.to(device)
    n_sample = inp.shape[0]
    t=0
    pred, theta = autoregressive_predict(model, inp[:, t:t+16], n_pred=target.shape[1], device=device)    
    print(theta.shape)
    all_theta.append(theta)

all_labels = [split] * len(all_theta)
all_theta = torch.stack(all_theta)
all_theta = all_theta.detach().cpu().numpy().squeeze(0)


In [ ]:

plt.scatter(all_theta[:, 0], all_theta[:, 1], c=ci.max(-1)[0], alpha=0.4, s=5)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label("Initial condition max")
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:

plt.scatter(all_theta[:, 0], all_theta[:, 1], c=ci.max(-1)[0] - ci.min(-1)[0], alpha=0.4, s=5)
plt.xlabel("theta_latent[0]")
plt.ylabel("theta_latent[1]")
plt.legend()
cbar = plt.colorbar()
cbar.set_label("Initial condition max")
plt.title("Theta latent 2D projection")
plt.tight_layout()
#plt.savefig(os.path.join(f"{results_dir}/{dataset_name}/plots/", f"test_pca_theta_vs_{pivot}.png"))
plt.show()
plt.close()

In [ ]:
# --- Function to extract Fourier statistics ---
def extract_fourier_statistics(signal):
    """
    Extracts Fourier-based statistics from a 1D signal.
    Adapt for 2D/3D signals (e.g., using np.fft.fft2, np.fft.fftn).
    """
    N = len(signal)
    if N == 0:
        return {
            'dominant_frequency': np.nan,
            'low_freq_power': np.nan,
            'high_freq_power': np.nan,
            'spectral_centroid': np.nan
        }

    # Perform FFT
    yf = np.fft.fft(signal)
    # Get the frequencies corresponding to the FFT result
    xf = np.fft.fftfreq(N, d=1) # d=1 assumes sampling interval of 1 (e.g., 1 unit per sample)

    # Consider only the positive frequencies for real signals (first half)
    half_N = N // 2
    xf_pos = xf[1:half_N + 1]
    yf_pos = np.abs(yf[1:half_N + 1]) # Magnitude spectrum

    if len(yf_pos) == 0: # Handle very short signals
        return {
            'dominant_frequency': np.nan,
            'low_freq_power': np.nan,
            'high_freq_power': np.nan,
            'spectral_centroid': np.nan
        }

    # 1. Dominant Frequency (frequency with highest magnitude)
    dominant_frequency_idx = np.argmax(yf_pos)
    dominant_frequency = xf_pos[dominant_frequency_idx]

    # 2. Total Power/Energy in Specific Frequency Bands
    # Define frequency bands (adjust these thresholds based on your data)
    low_freq_threshold = 0.1 # e.g., frequencies from 0 to 0.1
    high_freq_threshold = 0.4 # e.g., frequencies from 0.4 upwards

    low_freq_mask = (xf_pos >= 0) & (xf_pos <= low_freq_threshold)
    high_freq_mask = (xf_pos >= high_freq_threshold)

    # Power is magnitude squared
    power_spectrum = yf_pos**2

    low_freq_power = np.sum(power_spectrum[low_freq_mask])
    high_freq_power = np.sum(power_spectrum[high_freq_mask])

    # 3. Spectral Centroid
    # Weighted average of frequencies, where weights are magnitudes
    spectral_centroid = np.sum(xf_pos * yf_pos) / np.sum(yf_pos) if np.sum(yf_pos) > 0 else 0

    return {
        'dominant_frequency': dominant_frequency,
        'low_freq_power': low_freq_power,
        'high_freq_power': high_freq_power,
        'spectral_centroid': spectral_centroid
    }

In [ ]:
# --- Process all initial conditions ---
dominant_frequencies = []
low_freq_powers = []
high_freq_powers = []
spectral_centroids = []

for signal in ci:
    stats = extract_fourier_statistics(signal.squeeze(0))
    dominant_frequencies.append(stats['dominant_frequency'])
    low_freq_powers.append(stats['low_freq_power'])
    high_freq_powers.append(stats['high_freq_power'])
    spectral_centroids.append(stats['spectral_centroid'])

# Convert to numpy arrays for plotting
dominant_frequencies = np.array(dominant_frequencies)
low_freq_powers = np.array(low_freq_powers)
high_freq_powers = np.array(high_freq_powers)
spectral_centroids = np.array(spectral_centroids)


# --- Plotting the results ---

# Plot 1: theta_latent[0] vs. Dominant Frequency
plt.figure(figsize=(9, 7))
plt.scatter(all_theta[:, 0], all_theta[:, 1], alpha=0.6, s=15, c=dominant_frequencies, cmap='viridis')
plt.xlabel("theta_latent[0]")
plt.ylabel("Dominant Frequency of Initial Condition")
plt.title("Theta Latent[0] vs. Dominant Frequency")
plt.colorbar(label="Dominant Frequency")
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
plt.close()

# Plot 2: theta_latent[1] vs. Low Frequency Power
plt.figure(figsize=(9, 7))
plt.scatter(all_theta[:, 0], all_theta[:, 1], alpha=0.6, s=15, c=low_freq_powers, cmap='plasma')
plt.xlabel("theta_latent[1]")
plt.ylabel("Low Frequency Power of Initial Condition")
plt.title("Theta Latent[1] vs. Low Frequency Power")
plt.colorbar(label="Low Frequency Power")
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
plt.close()

# Plot 3: theta_latent[1] vs. Low Frequency Power
plt.figure(figsize=(9, 7))
plt.scatter(all_theta[:, 0], all_theta[:, 1], alpha=0.6, s=15, c=spectral_centroids, cmap='plasma')
plt.xlabel("theta_latent[1]")
plt.ylabel("Low Frequency Power of Initial Condition")
plt.title("Theta Latent[1] vs. Low Frequency Power")
plt.colorbar(label="Specral centroids")
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
plt.close()

# Plot 3: Pair Plot including Fourier statistics
import pandas as pd
import seaborn as sns

plot_data = pd.DataFrame({
    "theta_latent[0]": all_theta[:, 0],
    "theta_latent[1]": all_theta[:, 1],
    "Dominant_Frequency": dominant_frequencies,
    "Low_Freq_Power": low_freq_powers,
    "High_Freq_Power": high_freq_powers,
    "Spectral_Centroid": spectral_centroids
})

# You might want to sample if num_samples is very large for faster plotting
# plot_data_sample = plot_data.sample(n=min(2000, len(plot_data)))
# sns.pairplot(plot_data_sample) # Use plot_data_sample for large datasets

sns.pairplot(plot_data)
plt.suptitle("Pair Plot: Latent Dimensions vs. Fourier Statistics", y=1.02)
plt.tight_layout()
plt.show()
plt.close()

# Plot 4: Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(plot_data.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix: Latent Dimensions vs. Fourier Statistics")
plt.tight_layout()
plt.show()
plt.close()

# analysis the full theta and the layers

In [ ]:
def finite_difference_derivatives(x, dh):
    """
    Estimates the first and second derivatives of a vector using the central 
    finite difference method with periodic boundary conditions.
    
    Args:
        x (np.ndarray): The input vector.
        dh (float): The grid spacing.
        
    Returns:
        tuple: A tuple containing the estimated first and second derivatives.
    """
    # Use np.roll for efficient periodic boundary conditions
    x_plus_1 = np.roll(x, 1)  # x[i+1]
    x_minus_1 = np.roll(x, -1)  # x[i-1]
    
    # First derivative (central difference)
    # df/dh ~ (f(h+dh) - f(h-dh)) / (2*dh)
    dx_fd = (x_plus_1 - x_minus_1) / (2 * dh)
    
    # Second derivative (central difference)
    # d2f/dh2 ~ (f(h+dh) - 2*f(h) + f(h-dh)) / (dh^2)
    d2x_fd = (x_plus_1 - 2 * x + x_minus_1) / (dh**2)
    
    return dx_fd, d2x_fd

In [ ]:
class TemporalDatasetFixedCI(torch.utils.data.Dataset):
    def __init__(self, n_batches, batch_size, sub_x, sub_t, split="train", input_frames=16, output_frames=2,
                 L=16.0, nx=256, nt=100, T=10.0,
                 v_range=[0.01, 0.025, 0.05, 0.1, 0.5, 1.0], D_range=[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0],
                 fractal_degree=8, fractal_power=2, seed=None):
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.L = L
        self.nx = nx
        self.nt = nt
        self.T = T
        self.v_range = v_range
        self.D_range = D_range
        self.fractal_degree = fractal_degree
        self.fractal_power = fractal_power
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.u0 = []
        for _ in range(self.batch_size):
            fractaloid = Fractaloid(
                degree=self.fractal_degree,
                power=self.fractal_power,
                size=self.nx,
                patch_size=self.nx
            )
            u0 = fractaloid.generate(batch_size=1, seed=None).squeeze(0).numpy()
            u0 = (u0 - u0.mean()) / (u0.std() + 1e-8)
            self.u0.append(torch.from_numpy(u0))
            
        self.u0 = torch.stack(self.u0)

    def __len__(self):
        return len(self.u0)

    def __getitem__(self, idx):
    
        input_frames = self.input_frames
        batch_inputs = []
        batch_targets = []
        batch_v = []
        batch_d = []
        batch_init = []
        for v in self.v_range:
            for d in self.D_range:
                u0 = self.u0[idx]
                u_xt, x, t = advection_diffusion_analytical(
                    u0, L=self.L, v=v, D=d, nt=self.nt, T=self.T
                )
                u_xt = u_xt[::self.sub_t, ::self.sub_x]
                input = u_xt[:input_frames].copy()
                target = u_xt[input_frames: input_frames + self.output_frames].copy()
                
                batch_inputs.append(torch.from_numpy(input).unsqueeze(-2).float())
                batch_targets.append(torch.from_numpy(target).unsqueeze(-2).float())
                batch_v.append(v)
                batch_d.append(d)
                batch_init.append(u0)
                
        batch = {
            'input': torch.stack(batch_inputs),
            'target': torch.stack(batch_targets),
            'velocities': batch_v,
            'diffusivities': batch_d,
            'initial_conditions': torch.stack(batch_init)
        }
        return batch

In [ ]:
n_batches = 1  # or set as needed for your epoch size
batch_size = 128
split="test"
n_input_frames=16
n_output_frames=34
model.eval()

#composition_type="composition"
#composition_type="sum"

n= 100
advection_speeds = np.linspace(0.01, 1, n)
viscosities = [0.]*n


all_velocities = []
all_diffusivities = []
all_input = []
all_target = []
all_theta_latent = []
all_theta = []

train_ds = TemporalDatasetFixedCI(
        n_batches=n_batches,
        batch_size=batch_size,
        sub_x=sub_x,
        sub_t=sub_t,
        split=split,
        input_frames=n_input_frames,
        output_frames=n_output_frames,
        L=16.0,
        nx=256,
        nt=100,
        T=10.0,
        fractal_power=3.0,
        fractal_degree=256, # nx
        v_range=[0],#(0.01, 1.0),
        D_range=[0], #(0.001, 1.0),
    )


for advection_speed, viscosity in zip(advection_speeds, viscosities):

    train_ds.v_range=[advection_speed]
    train_ds.D_range=[viscosity]
    train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=1, prefetch_factor=1, pin_memory=True, shuffle=False)
    
    for batch in tqdm(train_loader):
        inp, target = batch["input"], batch["target"]
        inp = inp.squeeze(1)
        target = target.squeeze(1)
        print('inp', inp.shape, target.shape)
        all_velocities += batch["velocities"]
        all_diffusivities += batch["diffusivities"]
        
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
    
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)

    
    #predict
    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)
    
        # decode into 100k parameters
        theta = model.decode_theta(theta_latent, dim)
        pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames, predict_normed=False, metadata=metadata)
    rollout_error = relative_l2_error(pred, target[:, :n_output_frames]).item()
    
    print(f"Initial error", rollout_error)

    all_theta_latent.append(theta_latent)
    all_theta.append(theta)
    all_input.append(inp)
    all_target.append(target)


all_theta_latent = torch.stack(all_theta_latent)
all_theta = torch.stack(all_theta)
all_input = torch.stack(all_input)
all_target = torch.stack(all_target)

In [ ]:
dim=1
n,b,c = all_theta.shape
param_dict = dict(model.opnns[str(dim)].named_parameters())
batched_params_dict = vectors_to_parameters(rearrange(all_theta, "n b c-> (n b) c") , param_dict)

In [ ]:
# select first element, unsqueeze and repeat
x_inp = inp[:, 0].unsqueeze(0).repeat(n, 1, 1, 1)
x_inp = rearrange(x_inp, 'n b c h -> (n b) c h').unsqueeze(1)

In [ ]:
inp_labels = state_labels.clone()[None, ...].repeat(x_inp.shape[0], 1)

In [ ]:
with torch.no_grad():
    operator_output = vmap(functional_call, in_dims=(None, 0, 0))(model.opnns[str(dim)], batched_params_dict, (x_inp, inp_labels))

In [ ]:
operator_output = operator_output.squeeze(1).cpu().detach()
operator_output = rearrange(operator_output, "(n b) c h ->n b c h", n=n)

In [ ]:
idx=3
for i in range(0, n, 10):
    plt.plot(operator_output[i, idx].squeeze()/advection_speeds[i], label=f"Advection{advection_speeds[i]:2g}")
plt.title("f(x)/advection") 
plt.legend()

In [ ]:
dx_fd_est, d2x_fd_est = finite_difference_derivatives(inp[idx, 0].squeeze().cpu().detach().numpy(), 16/256)

In [ ]:
plt.plot(dx_fd_est)
plt.title("First derivative with finite difference")

In [ ]:
for k, v in zip(batched_params_dict.keys(), batched_params_dict.values()):
    batched_params_dict[k] = rearrange(batched_params_dict[k], '(n b) ... -> n b ...',n=n)

In [ ]:
batched_params_dict['downs.2.conv2.weight'].shape

In [ ]:
idx=0

In [ ]:
# Example: create a grid of 10×2 subplots
fig, axs = plt.subplots(nrows=10, ncols=2, figsize=(10, 20))

axs = axs.flatten()  # make it a flat list for easy indexing

for i in range(n):  # assume n <= 20
    ax = axs[i]
    #img = batched_params_dict['downs.2.conv2.weight'][i, idx, :, 0, :].cpu().detach()
    #ax.imshow(img, cmap='viridis')
    ax.plot(batched_params_dict['downs.2.conv2.weight'][i, idx, :, 0, 0].cpu().detach())
    ax.plot(batched_params_dict['downs.2.conv2.weight'][i, idx, :, 0, 1].cpu().detach())
    ax.plot(batched_params_dict['downs.2.conv2.weight'][i, idx, :, 0, 2].cpu().detach())
    ax.axis('off')  # hide ticks
    ax.set_title(f"Sample {i}", fontsize=8)

# hide any unused axes
for j in range(n, len(axs)):
    axs[j].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
theta_truncated = all_theta[..., :113000].cpu().detach()

In [ ]:

img = theta_truncated[i, idx].reshape(100, 1130)

plt.figure(figsize=(12, 6))
im = plt.imshow(img, cmap='seismic', aspect='auto')
plt.colorbar(im, fraction=0.02, pad=0.01, label='Intensity')
plt.title(f"Sample {i}, Index {idx}", fontsize=12)
plt.xlabel("Dimension 2")
plt.ylabel("Dimension 1")
plt.tight_layout()
plt.show()

In [ ]:
i=0
img = theta_truncated[i, idx].reshape(100, 1130)

plt.figure(figsize=(12, 6))
im = plt.imshow(img, cmap='seismic', aspect='auto')
plt.colorbar(im, fraction=0.02, pad=0.01, label='Intensity')
plt.title(f"Sample {i}, Index {idx}", fontsize=12)
plt.xlabel("Dimension 2")
plt.ylabel("Dimension 1")
plt.tight_layout()
plt.show()

In [ ]:
i=2
#img = theta_truncated[i, idx].reshape(100, 1130)
delta = (theta_truncated[i+1, idx] - theta_truncated[i,idx]).reshape(100, 1130)

plt.figure(figsize=(12, 6))
im = plt.imshow(delta, cmap='seismic', aspect='auto')
plt.colorbar(im, fraction=0.02, pad=0.01, label='Intensity')
plt.title(f"Sample {i}, Index {idx}", fontsize=12)
plt.xlabel("Dimension 2")
plt.ylabel("Dimension 1")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

cpt=0
for k, v in zip(batched_params_dict.keys(), batched_params_dict.values()):
    while v.ndim > 2:
        v = v.mean(-1)
    plt.plot(
        v.cpu().detach()[:, idx],
        label=f"mean-layer-{k}",
        linewidth=1,
        alpha=0.8
    )
    cpt+=1
    if cpt>10:
        break

# move legend outside the plot
plt.legend(
    bbox_to_anchor=(1.05, 1),  # to the right of the plot
    loc='upper left',
    fontsize=8,
    borderaxespad=0.
)

plt.xlabel("X-axis")
plt.ylabel("Mean value")
plt.title("Layer Means Over Something")
plt.tight_layout()  # adjust layout
plt.show()

In [ ]:
v.shape

In [ ]:
i=0
plt.plot(all_theta[i, idx].cpu().detach())

In [ ]:
i=0
delta = all_theta[1:] - all_theta[:-1]

In [ ]:
i=0
plt.plot(delta[i, idx].cpu().detach())

In [ ]:
i=8
plt.plot(delta[i, idx].cpu().detach())

In [ ]:
## try different analysis of this
index = 10
X = all_theta[:, index].cpu().detach()

In [ ]:
## 1. Simple statistics
plt.plot(X.mean(0))

In [ ]:
plt.plot(X.std(0))

In [ ]:
## PCA

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=2)
X_tf = pca.fit_transform(X)

In [ ]:
pca.explained_variance_ratio_

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid') # Apply a clean and modern style

plt.figure(figsize=(9, 7)) # Adjust figure size for better readability

categories = advection_speeds
scatter = plt.scatter(X_tf[:, 0], X_tf[:, 1],
                      c=categories,        # Use categories to color points
                      cmap='viridis',      # Choose a perceptually uniform colormap
                      s=120,               # Slightly larger markers
                      alpha=0.8,           # Good transparency for overlap
                      edgecolors='black',  # Black border for contrast
                      linewidths=0.7)      # Thicker border

# Add descriptive labels
plt.xlabel('First Component', fontsize=14)
plt.ylabel('Second Component', fontsize=14)
plt.title('Scatter Plot of First vs. Second Component', fontsize=16, pad=15)

# Add a legend for the categories if applicable
# This assumes 'categories' were used for 'c' and are discrete
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', label=f'Advection {i}',
                              markerfacecolor=scatter.cmap(scatter.norm(i)),
                              markersize=10, markeredgecolor='black')
                   for i in sorted(np.unique(categories))]
plt.legend(handles=legend_elements, title="Data Categories", bbox_to_anchor=(1.05, 1), loc='upper left')


plt.grid(True, linestyle='--', alpha=0.6) # Add a subtle grid
plt.axhline(0, color='gray', linestyle=':', linewidth=0.8) # Add horizontal line at 0
plt.axvline(0, color='gray', linestyle=':', linewidth=0.8) # Add vertical line at 0

# Adjust layout to prevent labels/title from overlapping
plt.tight_layout(rect=[0, 0, 0.88, 1]) # Adjust rect to make space for legend outside

# Display the plot
plt.show()


In [ ]:
n_batches = 1  # or set as needed for your epoch size
batch_size = 128
split="test"
n_input_frames=16
n_output_frames=34
model.eval()

#composition_type="composition"
#composition_type="sum"

n= 100
advection_speeds = [0.]*n
viscosities = np.linspace(0.001, 1, n)


all_velocities = []
all_diffusivities = []
all_input = []
all_target = []
all_theta_latent = []
all_theta = []

train_ds = TemporalDatasetFixedCI(
        n_batches=n_batches,
        batch_size=batch_size,
        sub_x=sub_x,
        sub_t=sub_t,
        split=split,
        input_frames=n_input_frames,
        output_frames=n_output_frames,
        L=16.0,
        nx=256,
        nt=100,
        T=10.0,
        fractal_power=3.0,
        fractal_degree=256, # nx
        v_range=[0],#(0.01, 1.0),
        D_range=[0], #(0.001, 1.0),
    )


for advection_speed, viscosity in zip(advection_speeds, viscosities):

    train_ds.v_range=[advection_speed]
    train_ds.D_range=[viscosity]
    train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=1, prefetch_factor=1, pin_memory=True, shuffle=False)
    
    for batch in tqdm(train_loader):
        inp, target = batch["input"], batch["target"]
        inp = inp.squeeze(1)
        target = target.squeeze(1)
        print('inp', inp.shape, target.shape)
        all_velocities += batch["velocities"]
        all_diffusivities += batch["diffusivities"]
        
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
    
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)

    
    #predict
    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)
    
        # decode into 100k parameters
        theta = model.decode_theta(theta_latent, dim)
        pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames, predict_normed=False, metadata=metadata)
    rollout_error = relative_l2_error(pred, target[:, :n_output_frames]).item()
    
    print(f"Initial error", rollout_error)

    all_theta_latent.append(theta_latent)
    all_theta.append(theta)
    all_input.append(inp)
    all_target.append(target)


all_theta_latent = torch.stack(all_theta_latent)
all_theta = torch.stack(all_theta)
all_input = torch.stack(all_input)
all_target = torch.stack(all_target)

In [ ]:
## try different analysis of this
index = 10
X = all_theta[:, index].cpu().detach()

In [ ]:
## 1. Simple statistics
plt.plot(X.mean(0))

In [ ]:
plt.plot(X.std(0))

In [ ]:
pca = PCA(n_components=2)
X_tf = pca.fit_transform(X)

In [ ]:
pca.explained_variance_ratio_

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid') # Apply a clean and modern style

plt.figure(figsize=(9, 7)) # Adjust figure size for better readability

categories = viscosities
scatter = plt.scatter(X_tf[:, 0], X_tf[:, 1],
                      c=categories,        # Use categories to color points
                      cmap='viridis',      # Choose a perceptually uniform colormap
                      s=120,               # Slightly larger markers
                      alpha=0.8,           # Good transparency for overlap
                      edgecolors='black',  # Black border for contrast
                      linewidths=0.7)      # Thicker border

# Add descriptive labels
plt.xlabel('First Component', fontsize=14)
plt.ylabel('Second Component', fontsize=14)
plt.title('Scatter Plot of First vs. Second Component', fontsize=16, pad=15)

# Add a legend for the categories if applicable
# This assumes 'categories' were used for 'c' and are discrete
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', label=f'Advection {i}',
                              markerfacecolor=scatter.cmap(scatter.norm(i)),
                              markersize=10, markeredgecolor='black')
                   for i in sorted(np.unique(categories))]
plt.legend(handles=legend_elements, title="Data Categories", bbox_to_anchor=(1.05, 1), loc='upper left')


plt.grid(True, linestyle='--', alpha=0.6) # Add a subtle grid
plt.axhline(0, color='gray', linestyle=':', linewidth=0.8) # Add horizontal line at 0
plt.axvline(0, color='gray', linestyle=':', linewidth=0.8) # Add vertical line at 0

# Adjust layout to prevent labels/title from overlapping
plt.tight_layout(rect=[0, 0, 0.88, 1]) # Adjust rect to make space for legend outside

# Display the plot
plt.show()


In [ ]:
dim=1
n,b,c = all_theta.shape
param_dict = dict(model.opnns[str(dim)].named_parameters())
batched_params_dict = vectors_to_parameters(rearrange(all_theta, "n b c-> (n b) c") , param_dict)

In [ ]:
# select first element, unsqueeze and repeat
x_inp = inp[:, 0].unsqueeze(0).repeat(n, 1, 1, 1)
x_inp = rearrange(x_inp, 'n b c h -> (n b) c h').unsqueeze(1)

In [ ]:
inp_labels = state_labels.clone()[None, ...].repeat(x_inp.shape[0], 1)

In [ ]:
with torch.no_grad():
    operator_output = vmap(functional_call, in_dims=(None, 0, 0))(model.opnns[str(dim)], batched_params_dict, (x_inp, inp_labels))

In [ ]:
operator_output = operator_output.squeeze(1).cpu().detach()
operator_output = rearrange(operator_output, "(n b) c h ->n b c h", n=n)

In [ ]:
idx=5

In [ ]:
dx_fd_est, d2x_fd_est = finite_difference_derivatives(inp[idx, 0].squeeze().cpu().detach().numpy(), 16/256)

In [ ]:

for i in range(0, n, 5):
    plt.plot(operator_output[i, idx].squeeze()/viscosities[i], label=f"Advection{viscosities[i]:2g}")
plt.title("f(x)/viscosity") 
plt.legend()

In [ ]:
plt.plot(d2x_fd_est)
plt.title("Second derivative with finite difference")

In [ ]:
x = inp[idx, 0].squeeze()

In [ ]:
delta = x[1:] - x[:-1]
delta2 = delta[1:] - delta[:-1]

In [ ]:
#plt.plot(inp[idx, 0].cpu().detach().squeeze())
#plt.plot(delta.cpu().detach().squeeze())
plt.plot(delta2.cpu().detach().squeeze())